# 通用伪源注入-恢复评估

本 Notebook 只读取已经冻结的评估产品，不重新训练或推理。经验 PSF 仅来自训练集，
检测阈值仅由验证集选择，最终完备度、纯度、F1、光度和位置误差仅来自测试集。

完整运行命令：`python scripts/evaluation/run_source_evaluation.py --model noise2noise --device cuda`。

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (start, *start.parents) if (path / "src" / "astr_ir").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("无法定位项目根目录")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

EVALUATION_ROOT = PROJECT_ROOT / "data" / "processed" / "evaluation" / "noise2noise"
SPLIT_PATH = PROJECT_ROOT / "data" / "processed" / "noise2noise" / "manifests" / "split_manifest.csv"
FIGURE_ROOT = PROJECT_ROOT / "figures" / "evaluation_output" / "noise2noise"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

from astr_ir.evaluation.visualization import (
    plot_completeness_purity,
    plot_empirical_psf,
    plot_photometric_accuracy,
    save_figure,
)

## 1. 数据职责隔离与经验 PSF

In [2]:
split = pd.read_csv(SPLIT_PATH, encoding="utf-8-sig", dtype={"sequence": str})
psf_diagnostics = pd.read_csv(
    EVALUATION_ROOT / "psf_training_diagnostics.csv",
    encoding="utf-8-sig",
    dtype={"sequence": str},
)
psf = np.asarray(fits.getdata(EVALUATION_ROOT / "empirical_psf.fits"), dtype=float)
display(split.groupby(["sequence", "split"]).size().rename("frames").to_frame())
display(psf_diagnostics.groupby(["sequence", "accepted"]).size().rename("cutouts").to_frame())
assert np.isclose(psf.sum(), 1.0, atol=2e-6)
assert (psf >= 0).all()
accepted_frames = set(psf_diagnostics.loc[psf_diagnostics["accepted"], "frame_id"])
assert all(split.set_index("frame_id").loc[frame_id, "split"] == "train" for frame_id in accepted_frames)
fig = plot_empirical_psf(psf)
save_figure(fig, FIGURE_ROOT / "empirical_psf.png")
plt.show()

frames
sequence split             
90000002 guard            4
         test            16
         train           48
         validation      12
90000003 guard            4
         test            16
         train           48
         validation      12

,,cutouts
sequence,accepted,
90000003,True,48


## 2. 仅使用验证集选择统一盲检阈值

In [3]:
calibration = pd.read_csv(
    EVALUATION_ROOT / "validation_threshold_calibration.csv", encoding="utf-8-sig"
)
with (EVALUATION_ROOT / "selected_threshold.json").open(encoding="utf-8") as handle:
    selected = json.load(handle)
display(pd.Series(selected).to_frame("value"))
display(calibration.loc[calibration["selected"]])
assert selected["selection_split"] == "validation"

,value
selected_threshold,4.0
selection_split,validation
target_minimum_purity,0.9
purity_gate_satisfied,True


,threshold,method,tp,fn,fp,completeness,purity,f1,minimum_method_purity,mean_method_f1,passes_purity_gate,selected
4,4.0,input,75,69,0,0.520833,1.000000,0.684932,0.952381,0.693343,True,True
5,4.0,output,80,64,4,0.555556,0.952381,0.701754,0.952381,0.693343,True,True


## 3. 测试集盲检完备度、纯度和95%置信区间

In [4]:
metrics = pd.read_csv(EVALUATION_ROOT / "metrics_by_snr.csv", encoding="utf-8-sig")
comparison = pd.read_csv(
    EVALUATION_ROOT / "paired_comparison_by_snr.csv", encoding="utf-8-sig"
)
display(metrics[[
    "method", "target_snr", "injected", "tp", "fn", "fp",
    "completeness", "completeness_ci_low", "completeness_ci_high",
    "purity", "purity_ci_low", "purity_ci_high", "f1",
]])
display(comparison)
fig = plot_completeness_purity(metrics)
save_figure(fig, FIGURE_ROOT / "completeness_purity.png")
plt.show()

,method,target_snr,injected,tp,fn,fp,completeness,completeness_ci_low,completeness_ci_high,purity,purity_ci_low,purity_ci_high,f1
0,input,2.0,256,4,252,0,0.015625,0.003906,0.031250,1.000000,1.000000,1.0,0.030769
1,output,2.0,256,6,250,3,0.023438,0.007812,0.039062,0.666667,0.333333,1.0,0.045283
2,input,3.0,256,34,222,0,0.132812,0.089844,0.179688,1.000000,1.000000,1.0,0.234483
3,output,3.0,256,39,217,2,0.152344,0.105469,0.199316,0.951220,0.874920,1.0,0.262626
4,input,4.0,256,94,162,0,0.367188,0.312500,0.421875,1.000000,1.000000,1.0,0.537143
5,output,4.0,256,110,146,3,0.429688,0.378906,0.480566,0.973451,0.939649,1.0,0.596206
6,input,5.0,256,187,69,1,0.730469,0.671875,0.777344,0.994681,0.984456,1.0,0.842342
7,output,5.0,256,197,59,3,0.769531,0.722656,0.816406,0.985000,0.966981,1.0,0.864035
8,input,7.0,256,249,7,0,0.972656,0.953125,0.988281,1.000000,1.000000,1.0,0.986139
9,output,7.0,256,249,7,2,0.972656,0.953125,0.988281,0.992032,0.980315,1.0,0.982249


,target_snr,injections,both_detected,neither_detected,input_only,output_only,paired_completeness_gain,gain_ci_low,gain_ci_high,mcnemar_exact_p,confidence_interval
0,2.0,256,4,250,0,2,0.007812,0.000000,0.019531,0.500000,stratified frame-cluster bootstrap 95%
1,3.0,256,34,217,0,5,0.019531,0.003906,0.035156,0.062500,stratified frame-cluster bootstrap 95%
2,4.0,256,94,146,0,16,0.062500,0.031250,0.097656,0.000031,stratified frame-cluster bootstrap 95%
3,5.0,256,187,59,0,10,0.039062,0.015625,0.066406,0.001953,stratified frame-cluster bootstrap 95%
4,7.0,256,249,7,0,0,0.000000,0.000000,0.000000,1.000000,stratified frame-cluster bootstrap 95%
5,10.0,256,251,5,0,0,0.000000,0.000000,0.000000,1.000000,stratified frame-cluster bootstrap 95%


## 4. 已恢复伪源的光度保真度

In [5]:
injections = pd.read_csv(EVALUATION_ROOT / "injection_recovery.csv", encoding="utf-8-sig")
recovered = injections.loc[injections["detected"]]
display(
    recovered.groupby(["method", "target_snr"])[
        ["relative_flux_error", "astrometric_error"]
    ].median()
)
fig = plot_photometric_accuracy(injections)
save_figure(fig, FIGURE_ROOT / "photometric_accuracy.png")
plt.show()

relative_flux_error  astrometric_error
method target_snr                                        
input  2.0                    1.241511           0.824818
       3.0                    0.469532           0.700921
       4.0                    0.126336           0.674172
       5.0                   -0.009714           0.448613
       7.0                   -0.062567           0.402037
       10.0                  -0.060014           0.377725
output 2.0                    0.944678           0.693932
       3.0                    0.264276           0.733595
       4.0                   -0.031075           0.665846
       5.0                   -0.126084           0.450381
       7.0                   -0.123530           0.402232
       10.0                  -0.087256           0.377725

SNR=2–3只保留越过检测阈值的正涨落，因而会出现明显正流量偏差；这是检测选择效应，
不能把仅对已检出对象计算的低SNR中位数直接解释为模型测光偏差。

## 5. 检测极限和未注入图像的新候选

In [6]:
summary = pd.read_csv(EVALUATION_ROOT / "evaluation_summary.csv", encoding="utf-8-sig")
unmodified = pd.read_csv(
    EVALUATION_ROOT / "unmodified_test_catalog.csv", encoding="utf-8-sig"
)
display(summary)
display(unmodified)

,metric,value
0,snr_at_50pct_completeness_input,4.365591
1,snr_at_50pct_completeness_output,4.206897
2,snr_limit_improvement_at_50pct_completeness,0.158695
3,snr_at_90pct_completeness_input,6.400000
4,snr_at_90pct_completeness_output,6.284615
5,snr_limit_improvement_at_90pct_completeness,0.115385
6,completeness_gain_snr_2,0.007812
7,completeness_gain_snr_3,0.019531
8,completeness_gain_snr_4,0.062500
9,completeness_gain_snr_5,0.039062


,frame_id,sequence,method,detection_id,x,y,score,flux,new_relative_to_input
0,90000002:065,90000002,output,0,991.0,276.0,6.024637,8015.203125,True
1,90000002:076,90000002,output,0,928.0,915.0,4.008276,5266.986816,True
2,90000003:069,90000003,output,0,990.0,59.0,4.247283,5559.662598,True


## 6. 严格验收

In [7]:
completed = subprocess.run(
    [sys.executable, "scripts/validation/validate_source_evaluation.py"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
print(completed.stdout)
if completed.stderr.strip():
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError("source evaluation strict validation failed")

selected threshold: 4.00
training PSF cutouts accepted: 48
test frames: 32
injection records: 3072
method  target_snr  injected  tp  fn  fp  completeness  completeness_ci_low  completeness_ci_high   purity  purity_ci_low  purity_ci_high  false_discovery_rate  false_positives_per_frame       f1  median_relative_flux_error  mad_relative_flux_error  median_astrometric_error_pixels  threshold                    confidence_interval
 input         2.0       256   4 252   0      0.015625             0.003906              0.031250 1.000000       1.000000             1.0              0.000000                    0.00000 0.030769                    1.241511                 0.057367                         0.824818        4.0 stratified frame-cluster bootstrap 95%
output         2.0       256   6 250   3      0.023438             0.007812              0.039062 0.666667       0.333333             1.0              0.333333                    0.09375 0.045283                    0.944678              

## 7. 全项目自动测试

In [8]:
completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
print(completed.stdout)
if completed.stderr.strip():
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError("pytest failed")

......................................                                   [100%]
38 passed in 9.65s



## 结论

冻结测试集显示 Noise2Noise 在SNR=4–5提供温和的盲检完备度提升，并保持约98%的纯度；
未注入测试图中出现3个仅输出候选，必须结合原始多帧一致性判断。当前结果是160帧数据上的基线，
新数据加入后应按完整80帧序列重新划分并扩大每个SNR的注入数量。